In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

DEBERTA_CKPT = "microsoft/deberta-v3-small"
ROBERTA_CKPT = "roberta-base"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
MAX_LEN = 512
LABEL_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

train_df = pd.read_csv(TRAIN_PATH) if os.path.exists(TRAIN_PATH) else None
test_df = pd.read_csv(TEST_PATH)

def format_mcq_input(row, instruction=None):
    base_prompt = f"Prompt: {row['prompt']}\nOptions:\nA: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}\nE: {row['E']}"
    if instruction:
        return f"{instruction} {base_prompt}"
    return base_prompt

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, instruction=None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.instruction = instruction

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = format_mcq_input(row, instruction=self.instruction)
        
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        item = {key: val.squeeze(0) for key, val in inputs.items()}
        return item

def get_probabilities(df, model_name, checkpoint_path, instruction=None):
    print(f"[{model_name}] Initializing tokenizer & loading model weights.")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, num_labels=5)
    model.to(DEVICE)
    model.eval()

    dataset = MCQDataset(df, tokenizer, MAX_LEN, instruction=instruction)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Inference via {model_name}"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            
            # DeBERTa-v3/RoBERTa compatibility check for token_type_ids
            inputs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                inputs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
                
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    
    return np.vstack(all_probs)


# Execution Pipelines for Questions 
print("\n Running Global Model Inference ")
# Get baseline probabilities for the entire test dataset
deberta_probs = get_probabilities(test_df, "DeBERTa", DEBERTA_CKPT)
roberta_probs = get_probabilities(test_df, "RoBERTa", ROBERTA_CKPT)

# Question 1 & 2 & 3 & 4 (Analysis on Row Index 25)
print("\n Analyzing Sample Row Index 25 ")
row_idx = 25
q1_deb_probs = deberta_probs[row_idx]
q1_max_idx = np.argmax(q1_deb_probs)
print(f"Q1 (DeBERTa Alone): Option {LABEL_MAP[q1_max_idx]}, Probability = {q1_deb_probs[q1_max_idx]:.4f}")

simple_avg_probs = (deberta_probs[row_idx] + roberta_probs[row_idx]) / 2
q2_max_idx = np.argmax(simple_avg_probs)
print(f"Q2 (Simple Ensemble): Highest Option = {LABEL_MAP[q2_max_idx]}")

weighted_avg_probs_row = (0.70 * deberta_probs[row_idx]) + (0.30 * roberta_probs[row_idx])
q3_max_idx = np.argmax(weighted_avg_probs_row)
print(f"Q3 (Weighted Ensemble): Highest Option = {LABEL_MAP[q3_max_idx]}")

# Top-3 Pred string for Kaggle
top3_indices = np.argsort(weighted_avg_probs_row)[::-1][:3]
top3_string = " ".join([LABEL_MAP[idx] for idx in top3_indices])
print(f"Q4 (Top-3 Prediction String): {top3_string}")

# Question 5
print("\n Generating Full Dataset Submission File ")
final_weighted_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)

submission_rows = []
for idx, row in test_df.iterrows():
    probs = final_weighted_probs[idx]
    sorted_indices = np.argsort(probs)[::-1][:3]
    pred_str = " ".join([LABEL_MAP[i] for i in sorted_indices])
    submission_rows.append({"id": row['id'], "prediction": pred_str})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)
print(f"Q5: Rows generated in submission.csv (excluding header): {len(submission_df)}")

# Question 6
print("\n Test-Time Augmentation Analysis ")
first_50_df = test_df.iloc[:50].copy()
tta_instruction = "Answer the following multiple-choice question carefully:"

# Inference on clean vs instructions
deberta_probs_50_orig = deberta_probs[:50]
deberta_probs_50_tta = get_probabilities(first_50_df, "DeBERTa-TTA", DEBERTA_CKPT, instruction=tta_instruction)

tta_combined_probs = (deberta_probs_50_orig + deberta_probs_50_tta) / 2

tta_changes = 0
for i in range(50):
    orig_top1 = np.argmax(deberta_probs_50_orig[i])
    tta_top1 = np.argmax(tta_combined_probs[i])
    if orig_top1 != tta_top1:
        tta_changes += 1
print(f"Q6: Count of first 50 rows producing a different Top-1 after TTA: {tta_changes}")

# Question 7, 8, & 9
print("\n Comparative Analysis")
changes_top1 = 0
positive_conf_gain = 0
top3_ranking_changes = 0

for i in range(100):
    deb_p = deberta_probs[i]
    ens_p = final_weighted_probs[i]
    
    # Q7 Checks
    if np.argmax(deb_p) != np.argmax(ens_p):
        changes_top1 += 1
        
    # Q8 Checks
    deb_conf = np.max(deb_p)
    ens_conf = np.max(ens_p)
    if (ens_conf - deb_conf) > 0:
        positive_conf_gain += 1
        
    # Q9 Checks
    deb_top3 = list(np.argsort(deb_p)[::-1][:3])
    ens_top3 = list(np.argsort(ens_p)[::-1][:3])
    if deb_top3 != ens_top3:
        top3_ranking_changes += 1

print(f"Q7: Rows with different Top-1 between DeBERTa and Ensemble: {changes_top1}")
print(f"Q8: Rows with a positive confidence gain: {positive_conf_gain}")
print(f"Q9: Rows with at least one change in Top-3 ranking layout: {top3_ranking_changes}")

# Question 10
def calculate_map3(predictions, labels):
    scores = []
    for pred, label in zip(predictions, labels):
        pred_list = pred.split()
        if label in pred_list:
            rank = pred_list.index(label) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)

if train_df is not None:
    print("\n Computing MAP@3 validation metrics ")
    val_subset = train_df.iloc[:100].copy()
    
    # Run predictions on the validation text
    val_deberta_probs = get_probabilities(val_subset, "Val-DeBERTa", DEBERTA_CKPT)
    val_roberta_probs = get_probabilities(val_subset, "Val-RoBERTa", ROBERTA_CKPT)
    val_ensemble_probs = (0.70 * val_deberta_probs) + (0.30 * val_roberta_probs)
    
    val_preds = []
    for idx in range(len(val_subset)):
        sorted_indices = np.argsort(val_ensemble_probs[idx])[::-1][:3]
        pred_str = " ".join([LABEL_MAP[i] for i in sorted_indices])
        val_preds.append(pred_str)
        
    true_labels = val_subset['answer'].values
    map3_score = calculate_map3(val_preds, true_labels)
    print(f"Q10: Final MAP@3 Score: {map3_score:.4f}")
else:
    print("\nTrain dataframe not found")


 Running Global Model Inference 
[DeBERTa] Initializing tokenizer & loading model weights.


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

[Val-RoBERTa] Initializing tokenizer & loading model weights.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Inference via Val-RoBERTa: 100%|██████████| 13/13 [00:02<00:00,  5.28it/s]


Q10: Final MAP@3 Score: 0.3283
